# Activity #1: Advanced Retrieval Evaluation

Evaluates 6 retriever methods using Ragas SDG and LangSmith

### YOUR CODE HERE
# Section 1: Imports
import os
from getpass import getpass
import pandas as pd
from operator import itemgetter
import time
import random
import pypdf

# LangChain
from langchain_community.document_loaders import PyPDFLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough

# Retrievers
from langchain_community.retrievers import BM25Retriever
from langchain.retrievers import (
    MultiQueryRetriever,
    ContextualCompressionRetriever,
    EnsembleRetriever,
    ParentDocumentRetriever
)
from langchain.retrievers.document_compressors import CohereRerank
from langchain.storage import InMemoryStore

# Ragas (0.2.10) 
from ragas import evaluate
from ragas.metrics import Faithfulness, ContextPrecision, ContextRecall, AnswerRelevancy, ContextEntityRecall
from ragas.testset import TestsetGenerator
from datasets import Dataset  # For Ragas 0.2.10 evaluation format

# LangSmith
from langsmith import Client

print("✅ All imports successful")

In [1]:
## YOUR CODE HERE
# Section 1: Imports
import os
from getpass import getpass
import pandas as pd
from operator import itemgetter
import time
import random
import pypdf

# LangChain
from langchain_community.document_loaders import PyPDFLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_qdrant import Qdrant
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough

# Retrievers
from langchain_community.retrievers import BM25Retriever
from langchain.retrievers import (
    MultiQueryRetriever,
    ContextualCompressionRetriever,
    EnsembleRetriever,
    ParentDocumentRetriever
)
from langchain.retrievers.document_compressors import CohereRerank
from langchain.storage import InMemoryStore

# Ragas (0.2.10) - 
from ragas import evaluate
from ragas.metrics import Faithfulness, ContextPrecision, ContextRecall, AnswerRelevancy, ContextEntityRecall
from ragas.testset import TestsetGenerator
from datasets import Dataset  # For Ragas 0.2.10 evaluation format

# LangSmith
from langsmith import Client

print("✅ All imports successful")


✅ All imports successful


In [2]:
# API Keys Setup
os.environ["OPENAI_API_KEY"] = getpass("Enter OpenAI API Key: ")
os.environ["COHERE_API_KEY"] = getpass("Enter Cohere API Key: ")
os.environ["LANGCHAIN_API_KEY"] = getpass("Enter LangChain/LangSmith API Key: ")

# Enable LangSmith tracing
os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_PROJECT"] = "Advanced_Retrieval_Evaluation"

langsmith_client = Client()
print("✅ API keys configured and LangSmith tracing enabled")


✅ API keys configured and LangSmith tracing enabled


In [3]:
# Section 2: Load PDF
pdf_path = "data/howpeopleuseai.pdf"
print(f"📄 Loading PDF: {pdf_path}")

loader = PyPDFLoader(pdf_path)
documents = loader.load()

print(f"✅ Loaded {len(documents)} pages from PDF")
print(f"   First page preview: {documents[0].page_content[:200]}...")

# Split for retrieval
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
chunks = text_splitter.split_documents(documents)
print(f"✅ Split into {len(chunks)} chunks for retrieval")


📄 Loading PDF: data/howpeopleuseai.pdf
✅ Loaded 64 pages from PDF
   First page preview: NBER WORKING PAPER SERIES
HOW PEOPLE USE CHATGPT
Aaron Chatterji
Thomas Cunningham
David J. Deming
Zoe Hitzig
Christopher Ong
Carl Yan Shan
Kevin Wadman
Working Paper 34255
http://www.nber.org/papers/...
✅ Split into 159 chunks for retrieval


In [ ]:
# Section 3: Golden Dataset Generation (Ragas SDG 0.2.10) - 10 Questions


from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper

generator_llm = LangchainLLMWrapper(ChatOpenAI(model="gpt-4o-mini"))
generator_embeddings = LangchainEmbeddingsWrapper(OpenAIEmbeddings(model="text-embedding-3-small"))

generator = TestsetGenerator(llm=generator_llm, embedding_model=generator_embeddings)
golden_dataset = generator.generate_with_langchain_docs(documents, testset_size=10)

# Convert to pandas for easy access
golden_df = golden_dataset.to_pandas()

print(f"✅ Generated {len(golden_df)} synthetic questions!")
print("\\n📋 Sample questions:")
for i in range(min(3, len(golden_df))):
    print(f"   {i+1}. {golden_df.iloc[i]['user_input']}")

# Display full dataset
golden_df


Applying HeadlinesExtractor:   0%|          | 0/22 [00:00<?, ?it/s]

Applying HeadlineSplitter:   0%|          | 0/64 [00:00<?, ?it/s]

unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to ap

Applying SummaryExtractor:   0%|          | 0/41 [00:00<?, ?it/s]

Property 'summary' already exists in node 'e5253b'. Skipping!
Property 'summary' already exists in node '11b21b'. Skipping!
Property 'summary' already exists in node 'fa172a'. Skipping!
Property 'summary' already exists in node 'c90e04'. Skipping!
Property 'summary' already exists in node 'b96bee'. Skipping!
Property 'summary' already exists in node '006b5a'. Skipping!
Property 'summary' already exists in node 'e57634'. Skipping!
Property 'summary' already exists in node '92289f'. Skipping!
Property 'summary' already exists in node 'd00d71'. Skipping!
Property 'summary' already exists in node '8fda67'. Skipping!
Property 'summary' already exists in node '5c2764'. Skipping!
Property 'summary' already exists in node 'e63f81'. Skipping!
Property 'summary' already exists in node 'aa21d7'. Skipping!
Property 'summary' already exists in node 'e12a88'. Skipping!
Property 'summary' already exists in node 'bd4b8b'. Skipping!
Property 'summary' already exists in node 'e9e960'. Skipping!
Property

...

Generating personas:   0%|          | 0/3 [00:00<?, ?it/s]

Generating Scenarios:   0%|          | 0/3 [00:00<?, ?it/s]

Generating Samples:   0%|          | 0/12 [00:00<?, ?it/s]

✅ Generated 12 synthetic questions!
\n📋 Sample questions:
   1. Wht is the role of Artificial Intelligenc in ChatGPT?
   2. How has OpenAI's ChatGPT influenced user behavior and market trends since its launch in November 2022?
   3. how much health care use chatgpt?


,user_input,reference_contexts,reference,synthesizer_name
0,Wht is the role of Artificial Intelligenc in C...,[Introduction ChatGPT launched in November 202...,"Artificial Intelligence, specifically in the f...",single_hop_specifc_query_synthesizer
1,How has OpenAI's ChatGPT influenced user behav...,[Introduction ChatGPT launched in November 202...,"Since its launch in November 2022, OpenAI's Ch...",single_hop_specifc_query_synthesizer
2,how much health care use chatgpt?,[Variation by Occupation Figure 23 presents va...,"Users in health care, categorized under other ...",single_hop_specifc_query_synthesizer
3,How education affect ChatGPT usage by occupation?,[Variation by Occupation Figure 23 presents va...,The context indicates that education is one of...,single_hop_specifc_query_synthesizer
4,What are the differences in ChatGPT usage betw...,[<1-hop>\n\nVariation by Occupation Figure 23 ...,The data shows that users in nonprofessional o...,multi_hop_abstract_query_synthesizer
5,How does ChatGPT's rapid adoption impact user ...,[<1-hop>\n\nIntroduction ChatGPT launched in N...,"ChatGPT, launched in November 2022, saw a rapi...",multi_hop_abstract_query_synthesizer
6,What are the differences in ChatGPT usage for ...,[<1-hop>\n\nVariation by Occupation Figure 23 ...,The differences in ChatGPT usage for work-rela...,multi_hop_abstract_query_synthesizer
7,What is the impact of the growth of ChatGPT on...,[<1-hop>\n\nConclusion This paper studies the ...,The growth of ChatGPT has had a significant im...,multi_hop_abstract_query_synthesizer
8,"By July 2025, how did the usage patterns of Ch...",[<1-hop>\n\nConclusion This paper studies the ...,"By July 2025, about 70% of ChatGPT consumer qu...",multi_hop_specific_query_synthesizer
9,What was the user engagement with ChatGPT by J...,[<1-hop>\n\nConclusion This paper studies the ...,"By July 2025, ChatGPT had achieved significant...",multi_hop_specific_query_synthesizer


In [8]:
# Section 4: Setup All 6 Retrievers
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

# 1. Naive
vectorstore = FAISS.from_documents(chunks, embeddings)
naive = vectorstore.as_retriever(search_kwargs={"k": 5})

# 2. BM25
bm25 = BM25Retriever.from_documents(chunks); bm25.k = 5

# 3. Multi-Query
multi_query = MultiQueryRetriever.from_llm(retriever=naive, llm=llm)

# 4. Parent Document
store = InMemoryStore()
parent_doc = ParentDocumentRetriever(
    vectorstore=FAISS.from_documents(RecursiveCharacterTextSplitter(chunk_size=400, chunk_overlap=50).split_documents(documents), embeddings),
    docstore=store,
    child_splitter=RecursiveCharacterTextSplitter(chunk_size=400, chunk_overlap=50),
    parent_splitter=RecursiveCharacterTextSplitter(chunk_size=2000, chunk_overlap=200)
)
parent_doc.add_documents(documents)

# 5. Contextual Compression (Cohere Rerank)
compression = ContextualCompressionRetriever(
    base_compressor=CohereRerank(model="rerank-english-v3.0", top_n=5),
    base_retriever=naive
)

# 6. Ensemble
ensemble = EnsembleRetriever(retrievers=[bm25, naive, multi_query], weights=[0.3, 0.4, 0.3])

retrievers = {"Naive": naive, "BM25": bm25, "Multi-Query": multi_query, 
              "Parent Document": parent_doc, "Contextual Compression": compression, "Ensemble": ensemble}

print(f"✅ All {len(retrievers)} retrievers ready")


/var/folders/lq/blff0y8x70d7sdk3bw6bzkvc0000gn/T/ipykernel_41529/3072977747.py:27: LangChainDeprecationWarning: The class `CohereRerank` was deprecated in LangChain 0.0.30 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-cohere package and should be used instead. To use it run `pip install -U :class:`~langchain-cohere` and import as `from :class:`~langchain_cohere import CohereRerank``.
  base_compressor=CohereRerank(model="rerank-english-v3.0", top_n=5),


✅ All 6 retrievers ready


In [9]:
# Section 5: Build RAG Chains with LangSmith Tracking
RAG_TEMPLATE = """You are a helpful AI assistant. Use the context to answer the question.
If you don't know, say so. Don't make up information.

Question: {question}
Context: {context}
Answer:"""

rag_prompt = ChatPromptTemplate.from_template(RAG_TEMPLATE)
rag_llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

def build_chain(retriever):
    return (
        {"context": itemgetter("question") | retriever, "question": itemgetter("question")}
        | RunnablePassthrough.assign(context=lambda x: "\\n\\n".join(d.page_content for d in x["context"]))
        | {"response": rag_prompt | rag_llm, "context": itemgetter("context")}
    )

rag_chains = {name: build_chain(r) for name, r in retrievers.items()}
print(f"✅ {len(rag_chains)} RAG chains ready with LangSmith tracking")


✅ 6 RAG chains ready with LangSmith tracking


In [ ]:
# Section 6: Evaluate All Retrievers with Ragas Metrics (0.2.10 API)
evaluator_llm = LangchainLLMWrapper(ChatOpenAI(model="gpt-4o-mini"))
metrics = [ContextPrecision(), ContextRecall(), AnswerRelevancy(), Faithfulness(), ContextEntityRecall()]

print(f"📊 Evaluating {len(retrievers)} retrievers with {len(metrics)} Ragas metrics...")
evaluation_results = {}

for name, chain in rag_chains.items():
    print(f"\\n{'='*50}\\nEvaluating: {name}\\n{'='*50}")
    start = time.time()
    
    # Build evaluation data (Ragas 0.2.10 format)
    # Required columns: question, answer, contexts, ground_truths, reference
    eval_data = {"question": [], "answer": [], "contexts": [], "ground_truths": [], "reference": []}
    
    # Use enumerate to get proper sequential count
    for idx in range(len(golden_df)):
        row = golden_df.iloc[idx]
        print(f"  Question {idx+1}/{len(golden_df)}...", end="\\r")
        try:
            question = row['user_input']
            reference = row.get('reference', '')
            
            result = chain.invoke({"question": question})
            
            eval_data["question"].append(question)
            eval_data["answer"].append(result["response"].content)
            eval_data["contexts"].append([result["context"]])
            eval_data["ground_truths"].append([reference] if reference else [question])
            eval_data["reference"].append(reference if reference else question)
            
            time.sleep(0.3)
        except Exception as e:
            print(f"\\n  ✗ Error on question {idx+1}: {e}")
            continue  # Skip failed questions
    
    print(f"\\n  Running Ragas evaluation...")
    from datasets import Dataset
    eval_dataset = Dataset.from_dict(eval_data)
    ragas_results = evaluate(eval_dataset, metrics=metrics, llm=evaluator_llm)
    elapsed = time.time() - start
    
    evaluation_results[name] = {"ragas": ragas_results, "latency": elapsed, "count": len(eval_data["question"])}
    print(f"  ✅ Complete ({elapsed:.1f}s) - Evaluated {len(eval_data['question'])}/{len(golden_df)} questions")

print("\\n" + "="*70)
print("📊 EVALUATION COMPLETE - QUICK SUMMARY")
print("="*70)

# Quick summary table
summary_data = []
for name, res in evaluation_results.items():
    ragas_result = res["ragas"]
    row = {
        "Retriever": name,
        "Questions": res["count"],
        "Latency (s)": f"{res['latency']:.1f}"
    }
    # Add first few metrics for quick view
    for metric in metrics[:3]:  # Show first 3 metrics
        if hasattr(ragas_result, metric.name):
            row[metric.name] = f"{getattr(ragas_result, metric.name):.3f}"
    summary_data.append(row)

summary_df = pd.DataFrame(summary_data)
print(summary_df.to_string(index=False))
print("\\n✅ Full detailed results in next cell...")


📊 Evaluating 6 retrievers with 5 Ragas metrics...
\n==================================================\nEvaluating: Naive\n==================================================
  Question 1/12...\r  Question 2/12...\r  Question 3/12...\r  Question 4/12...\r  Question 5/12...\r  Question 6/12...\r  Question 7/12...\r  Question 8/12...\r  Question 9/12...\r  Question 10/12...\r  Question 11/12...\r  Question 12/12...\r\n  Running Ragas evaluation...


Evaluating:   0%|          | 0/60 [00:00<?, ?it/s]

  ✅ Complete (115.2s): {'context_precision': 1.0000, 'context_recall': 0.7783, 'answer_relevancy': 0.8778, 'faithfulness': 0.9443, 'context_entity_recall': 0.3580}
\n==================================================\nEvaluating: BM25\n==================================================
  Question 1/12...\r  Question 2/12...\r  Question 3/12...\r  Question 4/12...\r  Question 5/12...\r  Question 6/12...\r  Question 7/12...\r  Question 8/12...\r  Question 9/12...\r  Question 10/12...\r  Question 11/12...\r  Question 12/12...\r\n  Running Ragas evaluation...


Evaluating:   0%|          | 0/60 [00:00<?, ?it/s]

  ✅ Complete (106.7s): {'context_precision': 1.0000, 'context_recall': 0.7535, 'answer_relevancy': 0.7932, 'faithfulness': 0.8699, 'context_entity_recall': 0.3271}
\n==================================================\nEvaluating: Multi-Query\n==================================================
  Question 1/12...\r  Question 2/12...\r  Question 3/12...\r  Question 4/12...\r  Question 5/12...\r  Question 6/12...\r  Question 7/12...\r  Question 8/12...\r  Question 9/12...\r  Question 10/12...\r  Question 11/12...\r  Question 12/12...\r\n  Running Ragas evaluation...


...

  ✅ Complete (136.7s): {'context_precision': 1.0000, 'context_recall': 0.7188, 'answer_relevancy': 0.8775, 'faithfulness': 0.8428, 'context_entity_recall': 0.3587}
\n==================================================\nEvaluating: Contextual Compression\n==================================================
  Question 1/12...\r  Question 2/12...\r  Question 3/12...\r  Question 4/12...\r  Question 5/12...\r  Question 6/12...\r  Question 7/12...\r  Question 8/12...\r  Question 9/12...\r  Question 10/12...\r  Question 11/12...\r\n  ✗ Error: status_code: 429, body: data=None id='b420e4bc-1e04-4970-8652-7122d7b8e7a0' message="You are using a Trial key, which is limited to 10 API calls / minute. You can continue to use the Trial key for free or upgrade to a Production key with higher rate limits at 'https://dashboard.cohere.com/api-keys'. Contact us on 'https://discord.gg/XW44jPfYJu' or email us at support@cohere.com with any questions"
  Question 12/12...\r\n  ✗ Error: status_code: 429, body: d

Evaluating:   0%|          | 0/50 [00:00<?, ?it/s]

  ✅ Complete (109.0s): {'context_precision': 0.9000, 'context_recall': 0.7881, 'answer_relevancy': 0.8575, 'faithfulness': 0.8020, 'context_entity_recall': 0.4451}
\n==================================================\nEvaluating: Ensemble\n==================================================
  Question 1/12...\r  Question 2/12...\r  Question 3/12...\r  Question 4/12...\r  Question 5/12...\r  Question 6/12...\r  Question 7/12...\r  Question 8/12...\r  Question 9/12...\r  Question 10/12...\r  Question 11/12...\r  Question 12/12...\r\n  Running Ragas evaluation...


Evaluating:   0%|          | 0/60 [00:00<?, ?it/s]

  ✅ Complete (164.0s): {'context_precision': 0.9167, 'context_recall': 0.8819, 'answer_relevancy': 0.8791, 'faithfulness': 0.9085, 'context_entity_recall': 0.3320}


In [ ]:
# Section 7: Results Compilation and Analysis
results_data = []
for name, res in evaluation_results.items():
    # Ragas 0.2.10: EvaluationResult has metric scores as attributes
    ragas_result = res["ragas"]
    row = {
        "Retriever": name, 
        "Latency (s)": res["latency"], 
        "Questions": res["count"]
    }
    
    # Extract metric scores from EvaluationResult
    for metric in metrics:
        metric_name = metric.name
        if hasattr(ragas_result, metric_name):
            row[metric_name] = getattr(ragas_result, metric_name)
    
    results_data.append(row)

results_df = pd.DataFrame(results_data).sort_values("Latency (s)")
print("\\n📊 FINAL RESULTS:")
print(results_df.to_string(index=False))

results_df.to_csv("retriever_evaluation_results.csv", index=False)
print("\\n✅ Saved to: retriever_evaluation_results.csv")

# Analysis
best = {col: results_df.loc[results_df[col].idxmax(), "Retriever"] 
        for col in results_df.columns if col not in ["Retriever", "Latency (s)", "Questions"]}

print("\\n🏆 Best by Metric:")
for metric, retriever in best.items():
    print(f"   {metric}: {retriever}")

print("\\n📝 RECOMMENDATION:")
print("""
Based on cost, latency, and performance:
- For best performance: Check scores above
- For best speed: Check latency column  
- For balanced approach: Consider all factors
""")

# LangSmith URLs
project_name = "Advanced_Retrieval_Evaluation"
print("\\n🔗 LangSmith Links:")
print(f"   📊 Project Dashboard: https://smith.langchain.com/o/default/projects/p/{project_name.replace('_', '-').lower()}")
print(f"   🔍 View All Traces: https://smith.langchain.com/")
print(f"   📁 Project: {project_name}")
print("\\n   💡 Tip: Use the LangSmith dashboard to:")
print("      - View detailed traces for each retriever")
print("      - Compare latency and token usage")
print("      - Debug retrieval quality issues")
print("      - Analyze cost breakdowns")

results_df


\n📊 FINAL RESULTS:
             Retriever  Latency (s)  Questions
                  BM25   106.687724         12
Contextual Compression   108.967348         10
                 Naive   115.187237         12
       Parent Document   136.651666         12
           Multi-Query   146.709074         12
              Ensemble   164.033424         12
\n✅ Saved to: retriever_evaluation_results.csv
\n🏆 Best by Metric:
\n📝 RECOMMENDATION:

Based on cost, latency, and performance:
- For best performance: Check scores above
- For best speed: Check latency column
- For balanced approach: Consider all factors

View traces in LangSmith: https://smith.langchain.com
Project: Advanced_Retrieval_Evaluation



,Retriever,Latency (s),Questions
1,BM25,106.687724,12
4,Contextual Compression,108.967348,10
0,Naive,115.187237,12
3,Parent Document,136.651666,12
2,Multi-Query,146.709074,12
5,Ensemble,164.033424,12



Summary: 

For the second activity, I evaluated multiple retrievers using a set of 12 questions generated from PDF documents. The testing process involved setting up six retrievers—BM25, Naive, Parent Document, Multi-Query, Contextual Compression, and Ensemble —to extract relevant context from the PDFs and generate answers. We leveraged LangChain to orchestrate the retrieval and answer generation, which allowed us to measure latency (time taken for retrieval and gerneration) and understand potential cost implications for each strategy in LangSmith. Also, each retriever’s output was evaluated using the Ragas script, measuring several key metrics:

- Context precision - Measures how much of the retrieved context is actually relevant (quality of retrieved info).

- Context Recall – How much relevant information from the source documents was retrieved.

- Faithfulness – How much of the generated answer is supported by the retrieved context.

- Answer Relevancy – Whether the answer directly addresses the question asked.

- Context Entity Recall – The fraction of key entities from the source documents that appear in the answer or are retrieved.

The results highlighted clear trade-offs between speed, context coverage, answer quality, and entity recall. BM25 achieved the lowest latency (106.688s) while maintaining perfect context precision, though it had slightly lower context recall (0.754) and the lowest answer relevancy (0.793), and it tied for last in context entity recall. The Naive retriever maintained perfect context precision and achieved the highest faithfulness (0.944), ensuring highly reliable answers, with moderate latency and context recall. Ensemble (Rank Fusion) delivered the best context recall (0.882), strong answer relevancy (0.879), and high faithfulness (0.909), though its latency was higher (164.033s). The Ensemble retriever combined multiple individual retrievers—Naive, BM25, and Multi-Query—weighted [0.3, 0.4, 0.3], giving BM25 slightly more influence in the final retrieved context. This hybrid approach proved effective, particularly in improving context recall over the other retrievers. Contextual Compression had the highest context entity recall (0.445) but was limited to 10 questions due to API rate limits, highlighting considerations around throughput and cost. Overall, these results demonstrate how different retriever strategies balance speed, accuracy, and comprehensiveness when working with PDFs, providing actionable insights for selecting retrievers in production pipelines using LangChain.

📊 **Retriever Evaluation Summary (Rounded & Highlighted)**

| Retriever                 | Latency (s) | Context Precision | Context Recall | Answer Relevancy | Faithfulness | Context Entity Recall | Questions          |
|---------------------------|------------|-----------------|----------------|-----------------|-------------|---------------------|------------------|
| BM25                      | 106.688 ✅  | 1.000 ✅         | 0.754          | 0.793           | 0.870       | 0.327               | 12               |
| Contextual Compression    | 108.967    | 0.900           | 0.788          | 0.858           | 0.802       | 0.445 ✅            | 10 (rate-limited)|
| Naive            | 115.187    | 1.000 ✅         | 0.778          | 0.878           | 0.944 ✅     | 0.358               | 12               |
| Parent Document           | 136.652    | 1.000 ✅         | 0.719          | 0.878           | 0.843       | 0.359               | 12               |
| Multi-Query               | 146.709    | 0.917           | 0.792          | 0.876           | 0.828       | 0.327               | 12               |
| Ensemble (Rank Fusion)    | 164.033    | 0.917           | 0.882 ✅        | 0.879 ✅         | 0.909      | 0.332               | 12               |
